In [0]:
# ===========================================
# Notebook Name:
# 01_ingest_event_results
#
# Purpose:
# Fetch tournament result API responses for
# multiple tournaments selected by the Ops
# fetch-target table.
#
# Input:
# pokemon.ops.result_fetch_targets
#
# Outputs:
# pokemon.bronze.event_result_raw
# pokemon.bronze.scrape_log (source_type='event_result')
#
# Bronze grain:
# One row per tournament_id and response
# version (response_hash).
#
# Idempotency:
# tournament_id + response_hash
# ===========================================

In [0]:
from datetime import datetime, timezone
import hashlib
import json
import time
import uuid

import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [ ]:
RESULT_FETCH_TARGETS_TABLE = (
    "pokemon.ops.result_fetch_targets"
)

EVENT_RESULT_RAW_TABLE = (
    "pokemon.bronze.event_result_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

RESULT_API_URL = (
    "https://players.pokemon-card.com/"
    "event_result_detail_search"
)

RESULT_REFERER_TEMPLATE = (
    "https://players.pokemon-card.com/"
    "event/detail/{tournament_id}/result"
)

# Page size per request. Tournaments with more
# results than this are paginated across multiple
# requests and combined into one canonical payload
# (see fetch_event_result).
RESULT_PAGE_SIZE = 100

# Safety cap on pages per tournament so a
# misbehaving API response can never cause an
# unbounded request loop.
MAX_PAGES = 50

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 1.0

MAX_FETCH_TARGETS = 500

# A 404 can mean either "this tournament will
# never publish results" or "results are not
# published yet". A single 404 is therefore kept
# in the retry queue (request_status="error") for
# this many days from the first time it was
# observed; only once a tournament has 404'd for
# at least this long is it promoted to the
# permanent request_status="not_found".
RETRY_WINDOW_DAYS = 7

FAIL_NOTEBOOK_IF_ANY_ERROR = True

print("Input :", RESULT_FETCH_TARGETS_TABLE)
print("Output:", EVENT_RESULT_RAW_TABLE)
print("Log   :", SCRAPE_LOG_TABLE)

In [ ]:
# -------------------------------------------
# One pipeline_run_id per notebook execution,
# shared by every scrape_log row this run writes,
# so a single Workflow run can be traced across
# tasks/notebooks. Distinct from the per-request
# request_id assigned to each tournament fetch
# below. Falls back to a fresh uuid when run
# interactively outside a Databricks Job (no
# upstream initialize_pipeline_run task to read
# from).
# -------------------------------------------
pipeline_run_id = dbutils.jobs.taskValues.get(
    taskKey="initialize_pipeline_run",
    key="pipeline_run_id",
    default=None,
    debugValue="manual-run",
)

PIPELINE_RUN_ID = (
    pipeline_run_id
    if pipeline_run_id
    else str(uuid.uuid4())
)

print("pipeline_run_id:", PIPELINE_RUN_ID)

In [0]:
targets_df = (
    spark.table(RESULT_FETCH_TARGETS_TABLE)
    .select(
        "tournament_id",
        "fetch_reason",
        "priority",
        "event_date",
    )
    .filter(
        F.col("tournament_id").isNotNull()
    )
    .dropDuplicates(
        ["tournament_id"]
    )
    .orderBy(
        F.col("priority").asc_nulls_last(),
        F.col("event_date").desc_nulls_last(),
        F.col("tournament_id").desc(),
    )
    .limit(
        MAX_FETCH_TARGETS
    )
)

target_count = targets_df.count()

print(
    "Tournament result fetch targets:",
    target_count,
)

display(targets_df)

In [0]:
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1.0,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=[
        "GET",
    ],
)

http_adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    http_adapter,
)

session.mount(
    "http://",
    http_adapter,
)

REQUEST_HEADERS = {
    "Accept": (
        "application/json, "
        "text/plain, */*"
    ),
    "Accept-Language": (
        "ja,en-US;q=0.9,en;q=0.8"
    ),
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/150.0.0.0 "
        "Safari/537.36"
    ),
    "X-Accept-Version": "v1",
}

In [0]:
def serialize_payload(
    payload_object,
) -> str:
    return json.dumps(
        payload_object,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def calculate_sha256(
    value: str,
) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [0]:
RESULT_LIST_KEYS = [
    "result",
    "results",
    "event_result",
    "event_results",
    "ranking",
    "rankings",
    "players",
]


def find_result_list(
    payload_object,
):
    if not isinstance(
        payload_object,
        dict,
    ):
        return []

    for key in RESULT_LIST_KEYS:
        value = payload_object.get(key)

        if isinstance(value, list):
            return value

    data = payload_object.get("data")

    if isinstance(data, dict):
        for key in RESULT_LIST_KEYS:
            value = data.get(key)

            if isinstance(value, list):
                return value

    return []

In [ ]:
def merge_result_items_into_payload(
    first_payload_object,
    all_result_items,
):
    merged_payload_object = dict(
        first_payload_object
    )

    for key in RESULT_LIST_KEYS:
        value = merged_payload_object.get(key)

        if isinstance(value, list):
            merged_payload_object[key] = (
                all_result_items
            )
            return merged_payload_object

    data = merged_payload_object.get("data")

    if isinstance(data, dict):
        merged_data = dict(data)

        for key in RESULT_LIST_KEYS:
            value = merged_data.get(key)

            if isinstance(value, list):
                merged_data[key] = (
                    all_result_items
                )
                merged_payload_object["data"] = (
                    merged_data
                )
                return merged_payload_object

    return merged_payload_object


def fetch_event_result(
    tournament_id: str,
):
    headers = dict(
        REQUEST_HEADERS
    )

    headers["Referer"] = (
        RESULT_REFERER_TEMPLATE.format(
            tournament_id=tournament_id
        )
    )

    all_result_items = []
    first_payload_object = None
    total_count = None
    last_response = None
    last_api_code = None
    total_elapsed_ms = 0
    page_offset = 0
    page_number = 0

    while True:
        page_number += 1

        if page_number > MAX_PAGES:
            print(
                "  WARNING: "
                f"tournament_id={tournament_id} "
                f"reached MAX_PAGES={MAX_PAGES} "
                "while paginating results; "
                "remaining results were not fetched."
            )
            break

        params = {
            "event_holding_id": tournament_id,
            "offset": page_offset,
            "per_page": RESULT_PAGE_SIZE,
        }

        started_at = time.perf_counter()

        response = session.get(
            RESULT_API_URL,
            params=params,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        total_elapsed_ms += int(
            (
                time.perf_counter()
                - started_at
            ) * 1000
        )

        response.raise_for_status()

        content_type = (
            response.headers
            .get(
                "Content-Type",
                "",
            )
            .lower()
        )

        if "application/json" not in content_type:
            raise ValueError(
                "Tournament result API did not "
                "return JSON. "
                f"tournament_id={tournament_id}, "
                f"status={response.status_code}, "
                f"content_type={content_type}, "
                f"body={response.text[:500]}"
            )

        try:
            payload_object = response.json()

        except ValueError as error:
            raise ValueError(
                "Tournament result response "
                "could not be parsed as JSON. "
                f"tournament_id={tournament_id}, "
                f"body={response.text[:500]}"
            ) from error

        api_code = (
            payload_object.get("code")
            if isinstance(
                payload_object,
                dict,
            )
            else None
        )

        if (
            api_code is not None
            and int(api_code) != 200
        ):
            raise ValueError(
                "Tournament result API returned "
                "a non-success code. "
                f"tournament_id={tournament_id}, "
                f"api_code={api_code}"
            )

        last_api_code = api_code
        last_response = response

        if first_payload_object is None:
            first_payload_object = payload_object

            raw_count = (
                payload_object.get("count")
                if isinstance(
                    payload_object,
                    dict,
                )
                else None
            )

            total_count = (
                int(raw_count)
                if raw_count is not None
                else None
            )

        page_items = find_result_list(
            payload_object
        )

        all_result_items.extend(
            page_items
        )

        page_is_full = (
            len(page_items) == RESULT_PAGE_SIZE
        )

        known_total_reached = (
            total_count is not None
            and len(all_result_items) >= total_count
        )

        has_more_pages = (
            page_is_full
            and not known_total_reached
        )

        if not has_more_pages:
            break

        # Rate-limit between pages of the same
        # tournament, not just between tournaments,
        # so a single large tournament cannot burst
        # requests against the site.
        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

        page_offset += RESULT_PAGE_SIZE

    merged_payload_object = (
        merge_result_items_into_payload(
            first_payload_object,
            all_result_items,
        )
    )

    payload = serialize_payload(
        merged_payload_object
    )

    response_hash = calculate_sha256(
        payload
    )

    return {
        "request_url": last_response.url,
        "response_status": (
            last_response.status_code
        ),
        "api_code": (
            int(last_api_code)
            if last_api_code is not None
            else None
        ),
        "payload": payload,
        "response_hash": response_hash,
        "result_count": len(
            all_result_items
        ),
        "page_count": page_number,
        "elapsed_ms": total_elapsed_ms,
    }

In [ ]:
# -------------------------------------------
# Existing response versions, scoped to just the
# tournament_ids targeted this run rather than a
# full-table scan of EVENT_RESULT_RAW_TABLE. This
# keeps the driver-side collect() bounded by
# MAX_FETCH_TARGETS regardless of how large the
# raw table grows historically.
# -------------------------------------------
target_tournament_ids = [
    row["tournament_id"]
    for row in (
        targets_df
        .select("tournament_id")
        .collect()
    )
]

existing_result_versions = {
    (
        row["tournament_id"],
        row["response_hash"],
    )
    for row in (
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .filter(
            F.col("tournament_id").isin(
                target_tournament_ids
            )
        )
        .select(
            "tournament_id",
            "response_hash",
        )
        .distinct()
        .collect()
    )
}

print(
    "Existing result response versions "
    "(targeted tournaments only):",
    len(existing_result_versions),
)

In [ ]:
raw_schema = StructType([
    StructField("tournament_id", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("payload", StringType(), False),
    StructField("payload_format", StringType(), False),
    StructField("response_hash", StringType(), False),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

scrape_log_schema = StructType([
    StructField("request_id", StringType(), False),
    StructField("pipeline_run_id", StringType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", StringType(), True),
    StructField("request_url", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("elapsed_ms", LongType(), True),
    StructField("records_found", IntegerType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("scraped_at", TimestampType(), False),
    StructField("ingestion_date", DateType(), False),
])

In [ ]:
# -------------------------------------------
# Look up, per tournament_id, the earliest
# scrape_log timestamp at which an event_result
# request for it returned HTTP 404. http_status
# is recorded on every HTTPError regardless of
# whether request_status ends up "error" or
# "not_found", so this lookup sees every 404 ever
# observed, even the ones still inside the retry
# window from a previous run.
# -------------------------------------------
first_404_seen_df = (
    spark.table(SCRAPE_LOG_TABLE)
    .filter(
        (F.col("source_type") == "event_result")
        & (F.col("http_status") == 404)
    )
    .groupBy("source_id")
    .agg(
        F.min("scraped_at").alias("first_404_at")
    )
)

first_404_seen_by_tournament_id = {}

for row in first_404_seen_df.collect():
    first_404_at = row["first_404_at"]

    if first_404_at.tzinfo is None:
        first_404_at = first_404_at.replace(
            tzinfo=timezone.utc
        )

    first_404_seen_by_tournament_id[
        row["source_id"]
    ] = first_404_at

print(
    "Tournaments with at least one prior 404:",
    len(first_404_seen_by_tournament_id),
)

In [ ]:
target_rows = targets_df.collect()

raw_rows = []
log_rows = []

success_count = 0
success_empty_count = 0
skipped_count = 0
error_count = 0
not_found_count = 0
retry_pending_404_count = 0

for index, target in enumerate(
    target_rows,
    start=1,
):
    tournament_id = target["tournament_id"]
    fetch_reason = target["fetch_reason"] or "unknown"

    request_id = str(uuid.uuid4())
    fetched_at = datetime.now(timezone.utc)

    print(
        f"[{index}/{target_count}] "
        f"tournament_id={tournament_id}, "
        f"reason={fetch_reason}"
    )

    try:
        result = fetch_event_result(
            tournament_id
        )

        result_version = (
            tournament_id,
            result["response_hash"],
        )

        is_new_response = (
            result_version
            not in existing_result_versions
        )

        if result["result_count"] == 0:
            request_status = "success_empty"
            success_empty_count += 1

        elif is_new_response:
            request_status = "success"
            success_count += 1

        else:
            request_status = "skipped_unchanged"
            skipped_count += 1

        if is_new_response:
            raw_rows.append({
                "tournament_id": tournament_id,
                "source_url": result["request_url"],
                "http_status": result["response_status"],
                "payload": result["payload"],
                "payload_format": "json",
                "response_hash": result["response_hash"],
                "scraped_at": fetched_at,
                "ingestion_date": fetched_at.date(),
            })

            existing_result_versions.add(
                result_version
            )

        log_rows.append({
            "request_id": request_id,
            "pipeline_run_id": PIPELINE_RUN_ID,
            "source_type": "event_result",
            "source_id": tournament_id,
            "request_url": result["request_url"],
            "http_status": result["response_status"],
            "status": request_status,
            "elapsed_ms": result["elapsed_ms"],
            "records_found": result["result_count"],
            "error_type": None,
            "error_message": None,
            "scraped_at": fetched_at,
            "ingestion_date": fetched_at.date(),
        })

        print(
            f"  status={request_status}, "
            f"result_count={result['result_count']}, "
            f"new_response={is_new_response}"
        )

    except requests.exceptions.HTTPError as error:
        is_404 = (
            error.response is not None
            and error.response.status_code == 404
        )

        # A 404 can mean either "this tournament will
        # never publish results" or "results are not
        # published yet". Only promote it to the
        # permanent not_found status once it has been
        # 404'ing for at least RETRY_WINDOW_DAYS; until
        # then it is recorded as "error" (so it stays in
        # the retry_error queue downstream) but tracked
        # separately from real errors below, since a
        # fresh 404 inside the retry window is an
        # expected outcome, not a failure.
        if is_404:
            first_404_at = (
                first_404_seen_by_tournament_id.get(
                    tournament_id
                )
            )

            if first_404_at is None:
                is_permanent_not_found = False
            else:
                days_since_first_404 = (
                    fetched_at - first_404_at
                ).total_seconds() / 86400

                is_permanent_not_found = (
                    days_since_first_404
                    >= RETRY_WINDOW_DAYS
                )
        else:
            is_permanent_not_found = False

        request_status = (
            "not_found"
            if is_permanent_not_found
            else "error"
        )

        if is_permanent_not_found:
            not_found_count += 1
        elif is_404:
            retry_pending_404_count += 1
        else:
            error_count += 1

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )[:4000]

        log_rows.append({
            "request_id": request_id,
            "pipeline_run_id": PIPELINE_RUN_ID,
            "source_type": "event_result",
            "source_id": tournament_id,
            "request_url": (
                f"{RESULT_API_URL}"
                f"?event_holding_id={tournament_id}"
            ),
            "http_status": (
                error.response.status_code
                if error.response is not None
                else None
            ),
            "status": request_status,
            "elapsed_ms": None,
            "records_found": 0,
            "error_type": type(error).__name__,
            "error_message": error_message,
            "scraped_at": fetched_at,
            "ingestion_date": fetched_at.date(),
        })

        print(
            f"  status={request_status}, error={error_message}"
        )

    except Exception as error:
        error_count += 1

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )[:4000]

        log_rows.append({
            "request_id": request_id,
            "pipeline_run_id": PIPELINE_RUN_ID,
            "source_type": "event_result",
            "source_id": tournament_id,
            "request_url": (
                f"{RESULT_API_URL}"
                f"?event_holding_id={tournament_id}"
            ),
            "http_status": None,
            "status": "error",
            "elapsed_ms": None,
            "records_found": 0,
            "error_type": type(error).__name__,
            "error_message": error_message,
            "scraped_at": fetched_at,
            "ingestion_date": fetched_at.date(),
        })

        print(
            f"  status=error, error={error_message}"
        )

    if index < target_count:
        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

print("Success count:", success_count)
print("Success empty count:", success_empty_count)
print("Skipped count:", skipped_count)
print("Not found count:", not_found_count)
print("Retry-pending 404 count:", retry_pending_404_count)
print("Error count:", error_count)

In [ ]:
if raw_rows:
    event_result_raw_df = (
        spark.createDataFrame(
            raw_rows,
            schema=raw_schema,
        )
    )

    # Defense-in-depth: the in-loop check against
    # existing_result_versions already skips known
    # (tournament_id, response_hash) pairs, but that
    # set was read at the start of this run. Anti-join
    # against the live table again right before writing
    # so a concurrent run that inserted a matching
    # version while this notebook was fetching cannot
    # create a duplicate response version.
    new_event_result_raw_df = (
        event_result_raw_df.alias("new")
        .join(
            spark.table(
                EVENT_RESULT_RAW_TABLE
            )
            .select(
                "tournament_id",
                "response_hash",
            )
            .alias("existing"),
            on=[
                "tournament_id",
                "response_hash",
            ],
            how="left_anti",
        )
    )

    inserted_count = (
        new_event_result_raw_df.count()
    )

    (
        new_event_result_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            EVENT_RESULT_RAW_TABLE
        )
    )

    print(
        "Inserted event_result_raw rows:",
        inserted_count,
    )

    skipped_by_anti_join = (
        len(raw_rows) - inserted_count
    )

    if skipped_by_anti_join > 0:
        print(
            f"  {skipped_by_anti_join} row(s) "
            "were already present (race with a "
            "concurrent run) and were not "
            "re-inserted."
        )

else:
    print(
        "No new event_result_raw rows to insert."
    )

In [0]:
if log_rows:
    scrape_log_df = (
        spark.createDataFrame(
            log_rows,
            schema=scrape_log_schema,
        )
    )

    (
        scrape_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            SCRAPE_LOG_TABLE
        )
    )

    print(
        "Inserted scrape_log rows:",
        len(log_rows),
    )

else:
    print(
        "No scrape_log rows to insert."
    )

In [ ]:
request_ids_this_run = [
    row["request_id"] for row in log_rows
]

tournament_ids_fetched_this_run = [
    row["tournament_id"] for row in raw_rows
]

if tournament_ids_fetched_this_run:
    display(
        spark.table(
            EVENT_RESULT_RAW_TABLE
        )
        .filter(
            F.col("tournament_id").isin(
                tournament_ids_fetched_this_run
            )
        )
        .orderBy(
            F.col("scraped_at").desc()
        )
    )
else:
    print("No new event_result_raw rows this run.")

In [ ]:
display(
    spark.table(
        SCRAPE_LOG_TABLE
    )
    .filter(
        F.col("request_id").isin(
            list(request_ids_this_run)
        )
    )
    .orderBy(
        F.col("scraped_at").desc()
    )
)

In [0]:
duplicate_versions_df = (
    spark.table(
        EVENT_RESULT_RAW_TABLE
    )
    .groupBy(
        "tournament_id",
        "response_hash",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = (
    duplicate_versions_df.count()
)

if duplicate_count > 0:
    display(
        duplicate_versions_df
    )

    raise ValueError(
        f"{duplicate_count} duplicate "
        "event result response versions "
        "detected"
    )

print(
    "Validation passed: "
    "no duplicate tournament response "
    "versions"
)

In [0]:
if (
    FAIL_NOTEBOOK_IF_ANY_ERROR
    and error_count > 0
):
    raise RuntimeError(
        "Some tournament result requests "
        "failed. "
        f"error_count={error_count}"
    )